# 📊 Data Pre-processing & Normalisasi — Dataset Marketplace UMKM Bogor

Notebook ini melakukan **pre-processing** dan **normalisasi** data dari file `Marketplace Ready For Labeling.xlsx`  
sehingga dataset siap digunakan untuk proses **training model klasifikasi** produk UMKM Bogor.

## Alur Pipeline
1. **Load & Eksplorasi Awal** — membaca data Excel dan melihat kondisi awal
2. **Pembersihan Data** — handle missing values, duplikat, outlier
3. **Text Pre-processing** — cleaning & normalisasi nama produk
4. **Encoding Fitur Kategorikal** — Label Encoding & One-Hot Encoding
5. **Normalisasi Fitur Numerik** — MinMaxScaler, Log Transform
6. **Feature Engineering** — fitur tambahan dari data yang ada
7. **TF-IDF Vectorization** — representasi teks nama produk
8. **Export Dataset** — simpan dataset final siap training

## 📦 0. Import Library

In [ ]:
import os, re, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
import joblib

try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
    SASTRAWI_AVAILABLE = True
    print('✅ Sastrawi tersedia — stemming & stopword removal aktif')
except ImportError:
    SASTRAWI_AVAILABLE = False
    print('⚠️  Sastrawi tidak ditemukan — jalankan: pip install PySastrawi')

print(f'\n✅ Library siap | pandas {pd.__version__} | numpy {np.__version__}')

---
## 📁 1. Load & Eksplorasi Data Awal

In [ ]:
FILE_EXCEL = 'Marketplace Ready For Labeling.xlsx'

df_raw = pd.read_excel(FILE_EXCEL)

print('=' * 55)
print(f'📄 File   : {FILE_EXCEL}')
print(f'   Baris  : {df_raw.shape[0]}')
print(f'   Kolom  : {df_raw.shape[1]}')
print('=' * 55)

In [ ]:
df_raw.head(5)

In [ ]:
df_raw.info()

In [ ]:
print('❓ Missing Values per kolom:')
missing = df_raw.isnull().sum()
pct = (missing / len(df_raw) * 100).round(2)
print(pd.DataFrame({'Count': missing, 'Pct (%)': pct})[missing > 0])

In [ ]:
df_raw.describe()

In [ ]:
os.makedirs('plots', exist_ok=True)

cat_cols = ['kategori', 'sub_kategori', 'marketplace', 'lokasi']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
palette = sns.color_palette('muted')

for i, col in enumerate(cat_cols):
    counts = df_raw[col].value_counts().head(10)
    axes[i].barh(counts.index, counts.values, color=palette[i])
    axes[i].set_title(f'Distribusi {col}', fontweight='bold')
    axes[i].set_xlabel('Jumlah')
    for bar, v in zip(axes[i].patches, counts.values):
        axes[i].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                     str(v), va='center', fontsize=9)

plt.suptitle('Distribusi Kolom Kategorikal (Top 10)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plots/distribusi_kategorikal.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
num_cols = ['harga_produk', 'jumlah_', 'rating']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(num_cols):
    axes[i].hist(df_raw[col].dropna(), bins=30, color=palette[i], edgecolor='white')
    axes[i].set_title(f'Distribusi {col}', fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frekuensi')

plt.suptitle('Distribusi Kolom Numerik (Sebelum Normalisasi)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/distribusi_numerik_raw.png', bbox_inches='tight', dpi=120)
plt.show()

---
## 🧹 2. Pembersihan Data

In [ ]:
df = df_raw.copy()
print(f'Shape awal: {df.shape}')

### 2.1 Rename Kolom

In [ ]:
# Rename 'jumlah_' → 'jumlah_terjual'
df.rename(columns={'jumlah_': 'jumlah_terjual'}, inplace=True)
print('✅ Kolom direname:', df.columns.tolist())

### 2.2 Handle Missing Values

In [ ]:
median_jumlah = df['jumlah_terjual'].median()
df['jumlah_terjual'].fillna(median_jumlah, inplace=True)

df['jumlah_terjual'] = df['jumlah_terjual'].round(0).astype(int)
print(f'✅ jumlah_terjual NaN → diisi median = {median_jumlah}')

df['nama_toko'].fillna('Tidak Diketahui', inplace=True)
print('✅ nama_toko NaN → diisi "Tidak Diketahui"')

print('\n✅ Missing values setelah handling:')
print(df.isnull().sum())

### 2.3 Hapus Duplikat

In [ ]:
n_before = len(df)
df.drop_duplicates(subset=['url_produk'], keep='first', inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'🗑️  Duplikat dihapus: {n_before - len(df)}')
print(f'✅ Shape setelah dedup: {df.shape}')

### 2.4 Filter Baris Anomali

In [ ]:
n_before = len(df)
df = df[df['sub_kategori'] != 'Perlu Review'].copy()
print(f'🗑️  Baris "Perlu Review" dihapus: {n_before - len(df)}')

n_before = len(df)
df = df[df['harga_produk'] > 0].copy()
df.reset_index(drop=True, inplace=True)
print(f'🗑️  Baris harga ≤ 0 dihapus: {n_before - len(df)}')
print(f'\n✅ Shape setelah filter: {df.shape}')

### 2.5 Standardisasi Kolom Lokasi

In [ ]:
def normalize_lokasi(loc):
    loc = str(loc).strip().lower()
    if re.search(r'\bbogor\b', loc): return 'Bogor'
    if re.search(r'jakarta', loc):   return 'Jakarta'
    if re.search(r'depok', loc):     return 'Depok'
    if re.search(r'bekasi', loc):    return 'Bekasi'
    if re.search(r'tangerang', loc): return 'Tangerang'
    return 'Lainnya'

df['lokasi_clean'] = df['lokasi'].apply(normalize_lokasi)
df['is_bogor']     = (df['lokasi_clean'] == 'Bogor').astype(int)

print('📍 Distribusi lokasi (setelah normalisasi):')
print(df['lokasi_clean'].value_counts())
print(f'\n✅ is_bogor: {df["is_bogor"].value_counts().to_dict()}')

---
## ✍️ 3. Text Pre-processing — Nama Produk

In [ ]:
if SASTRAWI_AVAILABLE:
    factory  = StemmerFactory()
    stemmer  = factory.create_stemmer()
    sw_fac   = StopWordRemoverFactory()
    stopwords_id = set(sw_fac.get_stop_words())
    custom_sw = {
        'khas','bogor','produk','original','ori','asli','baru','murah',
        'gratis','terjual','ready','stok','tersedia','buah','satuan',
        'pcs','pack','gr','ml','kg','cm','liter','set','new','best','oleh'
    }
    stopwords_id.update(custom_sw)
    print(f'✅ Total stopwords: {len(stopwords_id)}')
else:
    stopwords_id = {'dan','atau','di','ke','dari','untuk','dengan','yang',
                    'ini','itu','khas','bogor','oleh','adalah','juga','sudah'}

In [ ]:
def clean_text(text):
    if not isinstance(text, str): return ''
    text = text.lower()
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\b\d+\b', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def remove_stopwords(text, sw):
    tokens = [t for t in text.split() if t not in sw and len(t) > 1]
    return ' '.join(tokens)

def stem_text(text, stm):
    return ' '.join([stm.stem(t) for t in text.split()])


print('🔄 Membersihkan teks...')
df['nama_produk_clean'] = df['nama_produk'].apply(clean_text)
df['nama_produk_clean'] = df['nama_produk_clean'].apply(lambda x: remove_stopwords(x, stopwords_id))

if SASTRAWI_AVAILABLE:
    df['nama_produk_clean'] = df['nama_produk_clean'].apply(lambda x: stem_text(x, stemmer))
    print('✅ Stopword removal + stemming selesai')
else:
    print('✅ Basic stopword removal selesai (tanpa stemming)')


# Fallback: jika nama_produk_clean kosong setelah stopword removal,
# gunakan clean_text dari nama_produk asli (tanpa stopword removal)
empty_mask = df['nama_produk_clean'].str.strip() == ''
n_empty = empty_mask.sum()
if n_empty > 0:
    df.loc[empty_mask, 'nama_produk_clean'] = df.loc[empty_mask, 'nama_produk'].apply(clean_text)
    print(f'⚠️  {n_empty} nama_produk_clean kosong → fallback ke clean_text asli')

print('\n📋 Contoh hasil cleaning:')
df[['nama_produk', 'nama_produk_clean']].head(5)

In [ ]:
df['panjang_raw']   = df['nama_produk'].str.split().str.len()
df['panjang_clean'] = df['nama_produk_clean'].str.split().str.len()

print(f'Rata-rata kata sebelum : {df["panjang_raw"].mean():.1f}')
print(f'Rata-rata kata sesudah : {df["panjang_clean"].mean():.1f}')
print(f'Reduksi                : {(1 - df["panjang_clean"].mean()/df["panjang_raw"].mean())*100:.1f}%')

---
## 🔢 4. Encoding Fitur Kategorikal

In [ ]:
le_kategori     = LabelEncoder()
le_sub_kategori = LabelEncoder()
le_marketplace  = LabelEncoder()
le_lokasi       = LabelEncoder()

df['kategori_encoded']     = le_kategori.fit_transform(df['kategori'])
df['sub_kategori_encoded'] = le_sub_kategori.fit_transform(df['sub_kategori'])
df['marketplace_encoded']  = le_marketplace.fit_transform(df['marketplace'])
df['lokasi_encoded']       = le_lokasi.fit_transform(df['lokasi_clean'])

print('✅ Label Encoding selesai:')
for le, name in [(le_kategori,'kategori'),(le_marketplace,'marketplace'),
                  (le_lokasi,'lokasi'),(le_sub_kategori,'sub_kategori')]:
    print(f'  {name}: {dict(zip(le.classes_, range(len(le.classes_))))}')

In [ ]:
# One-Hot Encoding untuk Marketplace & Lokasi
ohe_marketplace = pd.get_dummies(df['marketplace'], prefix='mp')
ohe_lokasi_df   = pd.get_dummies(df['lokasi_clean'], prefix='lokasi')

df = pd.concat([df, ohe_marketplace, ohe_lokasi_df], axis=1)

print('✅ One-Hot Encoding selesai:')
print('  marketplace :', ohe_marketplace.columns.tolist())
print('  lokasi      :', ohe_lokasi_df.columns.tolist())

---
## 📐 5. Normalisasi Fitur Numerik

### 5.1 Capping Outlier (IQR Method)

In [ ]:
def cap_outlier_iqr(series, factor=1.5):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - factor*IQR, Q3 + factor*IQR
    n_out = ((series < lo) | (series > hi)).sum()
    return series.clip(lower=lo, upper=hi), lo, hi, n_out

for col in ['harga_produk', 'jumlah_terjual']:
    capped, lo, hi, n = cap_outlier_iqr(df[col])
    df[col] = capped
    print(f'📌 {col}: batas IQR [{lo:,.0f} – {hi:,.0f}] | {n} outlier di-cap')

### 5.2 Log Transformation

In [ ]:
df['harga_log']  = np.log1p(df['harga_produk'])
df['jumlah_log'] = np.log1p(df['jumlah_terjual'])

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
plots = [
    ('harga_produk',   'Harga Produk (Asli)',       'steelblue'),
    ('harga_log',      'Harga Produk (log1p)',       'teal'),
    ('jumlah_terjual', 'Jumlah Terjual (Asli)',      'coral'),
    ('jumlah_log',     'Jumlah Terjual (log1p)',     'tomato'),
]
for ax, (col, title, color) in zip(axes.flatten(), plots):
    ax.hist(df[col].dropna(), bins=30, color=color, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Frekuensi')

plt.suptitle('Efek Log Transformation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/log_transform_comparison.png', bbox_inches='tight', dpi=120)
plt.show()
print('✅ Log transformation selesai')

### 5.3 MinMax Scaling

In [ ]:
features_to_scale = ['harga_log', 'jumlah_log', 'rating']

scaler_minmax = MinMaxScaler()
scaled_vals   = scaler_minmax.fit_transform(df[features_to_scale])

for i, col in enumerate(features_to_scale):
    df[f'{col}_scaled'] = scaled_vals[:, i]

print('✅ MinMax Scaling [0–1] selesai:')
print(df[[f'{c}_scaled' for c in features_to_scale]].describe().round(4))

### 5.4 Standard Scaling (untuk model berbasis jarak/gradient)

In [ ]:
scaler_std   = StandardScaler()
std_vals     = scaler_std.fit_transform(df[features_to_scale])

for i, col in enumerate(features_to_scale):
    df[f'{col}_std'] = std_vals[:, i]

print('✅ Standard Scaling (Z-score) selesai:')
print(df[[f'{c}_std' for c in features_to_scale]].describe().round(4))

---
## ⚙️ 6. Feature Engineering

In [ ]:
# Revenue proxy: estimasi omzet = harga × jumlah terjual
df['revenue_proxy']     = df['harga_produk'] * df['jumlah_terjual']
df['revenue_proxy_log'] = np.log1p(df['revenue_proxy'])

# Popularity score = rating × log(jumlah+1)
df['popularity_score'] = df['rating'] * np.log1p(df['jumlah_terjual'])

# Binary flag per kategori utama
df['is_makanan']  = (df['kategori'] == 'Makanan').astype(int)
df['is_minuman']  = (df['kategori'] == 'Minuman').astype(int)
df['is_fashion']  = (df['kategori'] == 'Pakaian & Fashion').astype(int)
df['is_souvenir'] = (df['kategori'] == 'Aksesoris & Souvenir').astype(int)

# Harga tier: 0=Murah(<30rb), 1=Menengah(30–75rb), 2=Premium(>75rb)
df['harga_tier'] = pd.cut(
    df['harga_produk'],
    bins=[0, 30_000, 75_000, float('inf')],
    labels=[0, 1, 2]
).astype(int)

print('✅ Feature Engineering selesai.')
new_feats = ['revenue_proxy_log','popularity_score','is_makanan','is_minuman',
             'is_fashion','is_souvenir','harga_tier','is_bogor']
df[new_feats].describe().round(3)

---
## 📝 7. TF-IDF Vectorization Nama Produk

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=200,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    max_df=0.95
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df['nama_produk_clean'])
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

print(f'✅ TF-IDF Matrix: {tfidf_matrix.shape[0]} dokumen × {tfidf_matrix.shape[1]} fitur')
print(f'   Contoh fitur: {tfidf_feature_names[:20].tolist()}')

In [ ]:
mean_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).flatten()
top_idx    = mean_tfidf.argsort()[::-1][:20]
top_words  = tfidf_feature_names[top_idx]
top_scores = mean_tfidf[top_idx]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_words[::-1], top_scores[::-1], color=sns.color_palette('Blues_r', 20))
ax.set_xlabel('Rata-rata Skor TF-IDF')
ax.set_title('Top 20 Kata/Frasa — TF-IDF (nama produk)', fontweight='bold')
plt.tight_layout()
plt.savefig('plots/tfidf_top20.png', bbox_inches='tight', dpi=120)
plt.show()

---
## 📦 8. Menyusun Dataset Final

### 8.1 Pilih Fitur untuk Training

In [ ]:
NUMERICAL_FEATURES = [
    'harga_log_scaled',
    'jumlah_log_scaled',
    'rating_scaled',
    'revenue_proxy_log',
    'popularity_score',
]

CATEGORICAL_FEATURES = [
    'kategori_encoded',
    'sub_kategori_encoded',
    'marketplace_encoded',
    'lokasi_encoded',
]

OHE_FEATURES = (
    [c for c in df.columns if c.startswith('mp_')] +
    [c for c in df.columns if c.startswith('lokasi_') and
     c not in ('lokasi_encoded', 'lokasi_clean')]
)

BINARY_FEATURES = [
    'is_bogor', 'is_makanan', 'is_minuman', 'is_fashion', 'is_souvenir', 'harga_tier'
]

ALL_TABULAR_FEATURES = NUMERICAL_FEATURES + CATEGORICAL_FEATURES + OHE_FEATURES + BINARY_FEATURES

print('✅ Ringkasan fitur:')
print(f'  Numerik      : {len(NUMERICAL_FEATURES)} fitur → {NUMERICAL_FEATURES}')
print(f'  Kategorikal  : {len(CATEGORICAL_FEATURES)} fitur')
print(f'  One-Hot      : {len(OHE_FEATURES)} fitur → {OHE_FEATURES}')
print(f'  Biner/Tier   : {len(BINARY_FEATURES)} fitur')
print(f'  TF-IDF Teks  : {tfidf_matrix.shape[1]} fitur')
print(f'  ─────────────────────────────────────')
print(f'  Total Tabular: {len(ALL_TABULAR_FEATURES)} fitur')
print(f'  Total Lengkap: {len(ALL_TABULAR_FEATURES) + tfidf_matrix.shape[1]} fitur')

### 8.2 Gabungkan Fitur Tabular + TF-IDF (Sparse Matrix)

In [ ]:
X_tabular  = csr_matrix(df[ALL_TABULAR_FEATURES].values.astype(float))
X_combined = hstack([X_tabular, tfidf_matrix])

print(f'✅ X_combined (tabular + TF-IDF): {X_combined.shape}')

### 8.3 Export Dataset Final ke CSV

In [ ]:
os.makedirs('../data/processed', exist_ok=True)

COLS_EXPORT = (
    ['url_produk', 'nama_produk', 'nama_produk_clean',
     'kategori', 'sub_kategori', 'marketplace', 'lokasi', 'lokasi_clean', 'nama_toko',
     'harga_produk', 'jumlah_terjual', 'rating']
    + ALL_TABULAR_FEATURES
)

df_export = df[COLS_EXPORT].copy()
df_export.fillna(0, inplace=True)

OUTPUT_CSV = '../data/processed/dataset_preprocessed.csv'
# ── Pastikan tipe data sudah benar sebelum disimpan ──────────────────────────
df['harga_produk']   = df['harga_produk'].round(0).astype(int)
df['jumlah_terjual'] = df['jumlah_terjual'].round(0).astype(int)
df['rating']         = df['rating'].round(1)
# Recalculate derived features setelah rounding
import numpy as _np
df['popularity_score']  = (df['rating'] * _np.log1p(df['jumlah_terjual'])).round(4)
df['revenue_proxy_log'] = _np.log1p(df['harga_produk'] * df['jumlah_terjual']).round(4)
print('✅ Tipe data dikunci: harga_produk=int, jumlah_terjual=int, rating=float')

df_export.to_csv(OUTPUT_CSV, index=False)

print(f'✅ Dataset preprocessed disimpan → {OUTPUT_CSV}')
print(f'   Shape : {df_export.shape}')
df_export.head(3)

### 8.4 Simpan Preprocessors

In [ ]:
os.makedirs('preprocessors', exist_ok=True)

joblib.dump(le_kategori,      'preprocessors/le_kategori.joblib')
joblib.dump(le_sub_kategori,  'preprocessors/le_sub_kategori.joblib')
joblib.dump(le_marketplace,   'preprocessors/le_marketplace.joblib')
joblib.dump(le_lokasi,        'preprocessors/le_lokasi.joblib')
joblib.dump(scaler_minmax,    'preprocessors/scaler_minmax.joblib')
joblib.dump(scaler_std,       'preprocessors/scaler_std.joblib')
joblib.dump(tfidf_vectorizer, 'preprocessors/tfidf_vectorizer.joblib')

feature_meta = {
    'numerical_features'  : NUMERICAL_FEATURES,
    'categorical_features': CATEGORICAL_FEATURES,
    'ohe_features'        : OHE_FEATURES,
    'binary_features'     : BINARY_FEATURES,
    'all_tabular_features': ALL_TABULAR_FEATURES,
    'tfidf_features'      : tfidf_feature_names.tolist(),
    'total_features'      : len(ALL_TABULAR_FEATURES) + len(tfidf_feature_names)
}
joblib.dump(feature_meta, 'preprocessors/feature_metadata.joblib')

print('✅ Preprocessors disimpan ke folder preprocessors/:')
for f in sorted(os.listdir('preprocessors')):
    size_kb = os.path.getsize(f'preprocessors/{f}') / 1024
    print(f'   {f:40s} ({size_kb:.1f} KB)')

---
## 📊 9. Ringkasan & Verifikasi Akhir

In [ ]:
print('=' * 60)
print('📋 RINGKASAN PIPELINE PRE-PROCESSING')
print('=' * 60)
print(f'  Input            : {FILE_EXCEL}')
print(f'  Baris awal       : {df_raw.shape[0]}')
print(f'  Baris akhir      : {df.shape[0]}')
print(f'  Baris dihapus    : {df_raw.shape[0] - df.shape[0]}')
print()
print(f'  Fitur tabular    : {len(ALL_TABULAR_FEATURES)}')
print(f'  Fitur TF-IDF     : {tfidf_matrix.shape[1]}')
print(f'  Total fitur      : {len(ALL_TABULAR_FEATURES) + tfidf_matrix.shape[1]}')
print()
print('  Output:')
print('    📁 ../data/processed/dataset_preprocessed.csv')
print('    📁 preprocessors/ (8 file joblib)')
print('    📁 plots/         (3 visualisasi)')
print('=' * 60)

In [ ]:
# Verifikasi akhir
df_check = pd.read_csv('../data/processed/dataset_preprocessed.csv')
nan_total = df_check.isnull().sum().sum()

if nan_total == 0:
    print('✅ VERIFIKASI LULUS: Tidak ada missing value!')
else:
    print(f'⚠️  WARNING: {nan_total} missing value ditemukan!')
    print(df_check.isnull().sum()[df_check.isnull().sum() > 0])

print(f'\n📦 Dataset siap training:')
print(f'   Baris  : {df_check.shape[0]}')
print(f'   Kolom  : {df_check.shape[1]}')
df_check.head(3)

---
> **📌 Catatan:** Dataset ini **belum memiliki kolom `label`**.  
> Setelah proses labeling selesai, tambahkan kolom `label` ke `data_processed/dataset_preprocessed.csv`  
> kemudian gunakan file tersebut untuk training model.